# ⚠️ 2025 winning severity model — historical, AutoCarver **7.0.5**

**This notebook is a period artifact. Do not copy its AutoCarver API into a new
project.** It is the run that won [ENS *Challenge Data*
#161](https://challengedata.ens.fr/challenges/161) in 2025, kept exactly as it was
executed so the "then vs now" comparison in `RESULTS.md` is reproducible.

It still **imports and runs** on a current AutoCarver while behaving differently:
selectors now take `n_best_features=` plus a `SelectionConfig` rather than
positional per-type budgets, and `ProcessingConfig(n_jobs=...)` carves across a
process pool where this run was single-process.

For the current pipeline see **`amount_model_2026.ipynb`**.


# Loading Data & Processing

First, let's load the challenge's data and target and join them on ``ID``

In [1]:
import pandas as pd

data_path = "../data/"

# loading x_train
data = pd.read_csv(data_path + "train_input_Z61KlZo.csv")
data.set_index("ID", inplace=True)
print("x_train", data.shape)

# loading target
target = pd.read_csv(data_path + "train_output_DzPxaPY.csv")
target.set_index("ID", inplace=True)
print("y_train", target.shape)

# joining x_train and y_train
data = data.join(target.drop("ANNEE_ASSURANCE", axis=1))
print("data", data.shape)

<tmp>/ipykernel_17376\1250711303.py:6: DtypeWarning: Columns (16,17,29,30,31,126,128,129,132,133,135,138,371) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv(data_path+'train_input_Z61KlZo.csv')


x_train (383610, 373)
y_train (383610, 4)
data (383610, 376)


## Stratified Sampling

**Stratified Sampling** is applied to ensure same distribution between train (80%) and dev (20%) samples

Stratification is done using classes that maximize association with the number of claims per observation. This allows samples to be similarly distributed on claim frequencies.

In [2]:
from AutoCarver import BinaryCarver
from sklearn.model_selection import train_test_split

target_col = "CM"

# loading target carver
target_carver = BinaryCarver.load("model/cm_carver_freq_tschuprowt.json")

# discretizing the target
y_freq = target_carver.transform(data)[target_col]

# Train-test split
x_train, x_dev, y_train, y_dev = train_test_split(
    data, data[target_col], test_size=0.2, random_state=42, stratify=y_freq
)
print("y_train mean", y_train.mean())
print("y_dev mean", y_dev.mean())

y_train mean 182.78860026567781
y_dev mean 181.45375929980975


In [3]:
print("amount mean per class")
(
    y_train.groupby(target_carver.transform(x_train)[target_col]).mean().sort_index(),
    y_dev.groupby(target_carver.transform(x_dev)[target_col]).mean().sort_index(),
)

amount mean per class


(CM
 0.0         0.634508
 1.0       726.987101
 2.0      1158.781582
 3.0     10554.699356
 4.0    197900.752676
 Name: CM, dtype: float64,
 CM
 0.0         0.570218
 1.0       731.756333
 2.0      1155.480222
 3.0     10198.466203
 4.0    198281.136167
 Name: CM, dtype: float64)

In [4]:
print("amount median per class")
(
    y_train.groupby(target_carver.transform(x_train)[target_col]).median().sort_index(),
    y_dev.groupby(target_carver.transform(x_dev)[target_col]).median().sort_index(),
)

amount median per class


(CM
 0.0         0.000
 1.0       723.900
 2.0      1131.300
 3.0      5313.735
 4.0    156541.820
 Name: CM, dtype: float64,
 CM
 0.0         0.000
 1.0       714.250
 2.0      1149.000
 3.0      5892.000
 4.0    179814.445
 Name: CM, dtype: float64)

# Feature Engineering

The `Processor` class is used to build several new features:
* `ZONE` is converted to `ZONE_REGION`
* Copies of `ALTITUDE_xxx`, `IND_xxx`, `MEN_xxx` and `LOG_xxx` are converted to numerical features: `ALTITUDE_xxx_num`, `IND_xxx_num`, `MEN_xxx_num` and `LOG_xxx_num`
* Vetusty of buildings is computed: `LOG_VETUSTE`
* It learns the distribution of `LOG_TOT`, `LOG_VETUSTE`, `MEN_TOT`, `IND_TOT`, `IND_SNV`,`ALTITUDE_TOT` per `ZONE_REGION`on train sample. It's then used to compute the ratio of each sample to its mean per region (sort of a measure of divergence from the regional mean)
* Per observation, `CA_TOT` and `CA_MEAN` are computed as the sum and mean of `CA1`, `CA2` and `CA3`. `CA_TOT` and `CA_MEAN` are used to compute ratios with  `CA1`, `CA2` and `CA3`
* Open Source data from [Base de Données sur les Incendies de Forêts en France](https://bdiff.agriculture.gouv.fr/) are added: fire extinction rates per `ZONE` in 2023, total surfaces burnt in 2023 and in the 2016-2020 period per `ZONE`, surface burnt over surface of forest per `ZONE` in 2023. Those features are crossed with `NB_CASERNES` and `ZONE_VENT`
* Numerical `KAPITAL_xxx` columns are sumed and maxed out into `KAPITAL_SUM` and `KAPITAL_MAX`
* Temperature columns are crossed with one another
* Binary non-numerical columns are one hot encoded

In [5]:
import warnings
from utils.data_toolkit import Processor

warnings.simplefilter(action="ignore", category=FutureWarning)

proc = Processor()
x_train = proc.fit_transform(x_train)
x_dev = proc.transform(x_dev)
data = proc.transform(data)
data.shape, x_train.shape, x_dev.shape

((383610, 562), (306888, 562), (76722, 562))

# Feature Processing

## Sorting features per data type

* numerical features
* categorical features
* ordinal features (with their respective ordering)

In [ ]:

# get the columns that are categorical
categorical_columns = x_train.select_dtypes(include=["object"]).columns

# get the columns that are numerical
numerical_columns = x_train.select_dtypes(include=["int64", "float64"]).columns

# getting ordinal columns
ordinals = [
    "NB_CASERNES",
    "BDTOPO_BAT_MAX_HAUTEUR",
    "HAUTEUR_MAX",
    "HAUTEUR",
    "BDTOPO_BAT_MAX_HAUTEUR_MAX",
    "MEN_SURF",
    "IND_SNV",
    "IND_INC",
    "IND_Y9",
    "IND_0_Y1",
    "IND",
    "LOG_SOC",
    "LOG_INC",
    "LOG_APA3",
    "LOG_AVA1",
    "MEN_MAIS",
    "MEN_COLL",
    "MEN_FMP",
    "MEN_PROP",
    "MEN_PAUV",
    "MEN",
    "COEFASS",
]
ordinals += [
    "DISTANCE_111",
    "DISTANCE_112",
    "DISTANCE_121",
    "DISTANCE_122",
    "DISTANCE_123",
    "DISTANCE_124",
    "DISTANCE_131",
    "DISTANCE_132",
    "DISTANCE_133",
    "DISTANCE_141",
    "DISTANCE_142",
    "DISTANCE_211",
    "DISTANCE_212",
    "DISTANCE_213",
    "DISTANCE_221",
    "DISTANCE_222",
    "DISTANCE_223",
    "DISTANCE_231",
    "DISTANCE_242",
    "DISTANCE_243",
    "DISTANCE_244",
    "DISTANCE_311",
    "DISTANCE_312",
    "DISTANCE_313",
    "DISTANCE_321",
    "DISTANCE_322",
    "DISTANCE_323",
    "DISTANCE_324",
    "DISTANCE_331",
    "DISTANCE_332",
    "DISTANCE_333",
    "DISTANCE_334",
    "DISTANCE_335",
    "DISTANCE_411",
    "DISTANCE_412",
    "DISTANCE_421",
    "DISTANCE_422",
    "DISTANCE_423",
    "DISTANCE_511",
    "DISTANCE_512",
    "DISTANCE_521",
    "DISTANCE_522",
    "DISTANCE_523",
    "PROPORTION_11",
    "PROPORTION_12",
    "PROPORTION_13",
    "PROPORTION_14",
    "PROPORTION_21",
    "PROPORTION_22",
    "PROPORTION_23",
    "PROPORTION_24",
    "PROPORTION_31",
    "PROPORTION_32",
    "PROPORTION_33",
    "PROPORTION_41",
    "PROPORTION_42",
    "PROPORTION_51",
    "PROPORTION_52",
    "MEN_1IND",
    "MEN_5IND",
    "LOG_A1_A2",
    "LOG_A2_A3",
    "IND_Y1_Y2",
    "IND_Y2_Y3",
    "IND_Y3_Y4",
    "IND_Y4_Y5",
    "IND_Y5_Y6",
    "IND_Y6_Y7",
    "IND_Y7_Y8",
    "IND_Y8_Y9",
    "DISTANCE_1",
    "DISTANCE_2",
    "ALTITUDE_1",
    "ALTITUDE_2",
    "ALTITUDE_3",
    "ALTITUDE_4",
    "ALTITUDE_5",
    "NBJTX25_MM_A",
    "NBJTX25_MMAX_A",
    "NBJTX25_MSOM_A",
    "NBJTX0_MM_A",
    "NBJTX0_MMAX_A",
    "NBJTX0_MSOM_A",
    "NBJTXI27_MM_A",
    "NBJTXI27_MMAX_A",
    "NBJTXI27_MSOM_A",
    "NBJTXS32_MM_A",
    "NBJTXS32_MMAX_A",
    "NBJTXS32_MSOM_A",
    "NBJTXI20_MM_A",
    "NBJTXI20_MMAX_A",
    "NBJTXI20_MSOM_A",
    "NBJTX30_MM_A",
    "NBJTX30_MMAX_A",
    "NBJTX30_MSOM_A",
    "NBJTX35_MM_A",
    "NBJTX35_MMAX_A",
    "NBJTX35_MSOM_A",
    "NBJTN10_MM_A",
    "NBJTN10_MMAX_A",
    "NBJTN10_MSOM_A",
    "NBJTNI10_MM_A",
    "NBJTNI10_MMAX_A",
    "NBJTNI10_MSOM_A",
    "NBJTN5_MM_A",
    "NBJTN5_MMAX_A",
    "NBJTN5_MSOM_A",
    "NBJTNS25_MM_A",
    "NBJTNS25_MMAX_A",
    "NBJTNS25_MSOM_A",
    "NBJTNI15_MM_A",
    "NBJTNI15_MMAX_A",
    "NBJTNI15_MSOM_A",
    "NBJTNI20_MM_A",
    "NBJTNI20_MMAX_A",
    "NBJTNI20_MSOM_A",
    "NBJTNS20_MM_A",
    "NBJTNS20_MMAX_A",
    "NBJTNS20_MSOM_A",
    "NBJTMS24_MM_A",
    "NBJTMS24_MMAX_A",
    "NBJTMS24_MSOM_A",
    "TAMPLIAB_VOR_MM_A",
    "TAMPLIAB_VOR_MMAX_A",
    "TAMPLIM_VOR_MM_A",
    "TAMPLIM_VOR_MMAX_A",
    "TM_VOR_MM_A",
    "TM_VOR_MMAX_A",
    "TMM_VOR_MM_A",
    "TMM_VOR_MMAX_A",
    "TMMAX_VOR_MM_A",
    "TMMAX_VOR_MMAX_A",
    "TMMIN_VOR_MM_A",
    "TMMIN_VOR_MMAX_A",
    "TN_VOR_MM_A",
    "TN_VOR_MMAX_A",
    "TNAB_VOR_MM_A",
    "TNAB_VOR_MMAX_A",
    "TNMAX_VOR_MM_A",
    "TNMAX_VOR_MMAX_A",
    "TX_VOR_MM_A",
    "TX_VOR_MMAX_A",
    "TXAB_VOR_MM_A",
    "TXAB_VOR_MMAX_A",
    "TXMIN_VOR_MM_A",
    "TXMIN_VOR_MMAX_A",
    "NBJFF10_MM_A",
    "NBJFF10_MMAX_A",
    "NBJFF10_MSOM_A",
    "NBJFF16_MM_A",
    "NBJFF16_MMAX_A",
    "NBJFF16_MSOM_A",
    "NBJFF28_MM_A",
    "NBJFF28_MMAX_A",
    "NBJFF28_MSOM_A",
    "NBJFXI3S10_MM_A",
    "NBJFXI3S10_MMAX_A",
    "NBJFXI3S10_MSOM_A",
    "NBJFXI3S16_MM_A",
    "NBJFXI3S16_MMAX_A",
    "NBJFXI3S16_MSOM_A",
    "NBJFXI3S28_MM_A",
    "NBJFXI3S28_MMAX_A",
    "NBJFXI3S28_MSOM_A",
    "NBJFXY8_MM_A",
    "NBJFXY8_MMAX_A",
    "NBJFXY8_MSOM_A",
    "NBJFXY10_MM_A",
    "NBJFXY10_MMAX_A",
    "NBJFXY10_MSOM_A",
    "NBJFXY15_MM_A",
    "NBJFXY15_MMAX_A",
    "NBJFXY15_MSOM_A",
    "FFM_VOR_MM_A",
    "FFM_VOR_MMAX_A",
    "FXI3SAB_VOR_MM_A",
    "FXI3SAB_VOR_MMAX_A",
    "FXIAB_VOR_MM_A",
    "FXIAB_VOR_MMAX_A",
    "FXYAB_VOR_MM_A",
    "FXYAB_VOR_MMAX_A",
    "FFM_VOR_COM_MM_A_Y",
    "FFM_VOR_COM_MMAX_A_Y",
    "FXI3SAB_VOR_COM_MM_A_Y",
    "FXI3SAB_VOR_COM_MMAX_A_Y",
    "NBJRR50_MM_A",
    "NBJRR50_MMAX_A",
    "NBJRR50_MSOM_A",
    "NBJRR1_MM_A",
    "NBJRR1_MMAX_A",
    "NBJRR1_MSOM_A",
    "NBJRR5_MM_A",
    "NBJRR5_MMAX_A",
    "NBJRR5_MSOM_A",
    "NBJRR10_MM_A",
    "NBJRR10_MMAX_A",
    "NBJRR10_MSOM_A",
    "NBJRR30_MM_A",
    "NBJRR30_MMAX_A",
    "NBJRR30_MSOM_A",
    "NBJRR100_MM_A",
    "NBJRR100_MMAX_A",
    "NBJRR100_MSOM_A",
    "RR_VOR_MM_A",
    "RR_VOR_MMAX_A",
    "RRAB_VOR_MM_A",
    "RRAB_VOR_MMAX_A",
]
# ordinals += ["AN_EXERC"]
ordinals += ["TAILLE1", "TAILLE2"]
ordinal_columns = {
    col: list(data[col].value_counts().sort_index().index)
    for col in ordinals
    if col in data.columns
}
ordinal_columns["PROPORTION_32"] += ["10. > 90"]
ordinal_columns.update(
    {
        "CARACT4": [
            "absence de surface",
            "Surface de moins d",
            "Surface entre 501",
            "Surface entre 1001",
            "Surface entre 1501",
            "Surface de plus de",
        ],
        "SURFACE4": [
            "0",
            "500",
            "1000",
            "1500",
            "2000",
            "2500",
            "3000",
            "3500",
            "4000",
            "4500",
            "5000",
            "5500",
            "6000",
            "6500",
            "7000",
            "7000+",
        ],
        "SURFACE6": [
            "0",
            "500",
            "1000",
            "1500",
            "2000",
            "2500",
            "3000",
            "3500",
            "4000",
            "4500",
            "5000",
            "5500",
            "6000",
            "6500",
            "7000",
            "7000+",
        ],
        "total_surface_2023": [
            "Aucun feu",
            "<10ha",
            "10-20ha",
            "20-50ha",
            "50-100ha",
            "100-200ha",
            ">200ha",
        ],
        "total_surface_5y": [
            "Aucun feu",
            "<10ha",
            "10-20ha",
            "20-50ha",
            "50-100ha",
            "100-200ha",
            ">200ha",
        ],
        "surface_over_forest": [
            "Absence de feu",
            "<0.05",
            "0.05-0.1",
            "0.1-0.2",
            "0.5-2",
        ],
        "fire_extinction_rates": ["Aucun feu", "<50%", "50-70%", "70-85%", ">85%"],
    }
)

# get the columns that are to be removed
to_remove = target.columns.tolist() + [target_col]
to_remove += [c for c in data.columns if "MMSOM" in c]
to_remove += [
    "DEROG3",
    "DEROG13",
    "DEROG16",
    "DEROG8",
    "DEROG14",
    "TARGET",
]  # no values
to_remove += [
    "DEROG13_formatted",
    "DEROG8_formatted",
    "DEROG3_formatted",
    "DEROG16_formatted",
    "DEROG14_formatted",
]
to_remove += ["IND_Y1_Y2_num", "IND_INC_num"]

# removing columns
categorical_columns = [
    col
    for col in categorical_columns
    if col not in to_remove and col not in ordinal_columns
]
categorical_columns += ["TYPERS"]
numerical_columns = [
    col
    for col in numerical_columns
    if col not in to_remove
    and col not in ordinal_columns
    and col not in categorical_columns
]
print(
    len(categorical_columns),
    len(numerical_columns),
    len(ordinal_columns),
    len(categorical_columns) + len(numerical_columns) + len(ordinal_columns),
)

89 221 238 548


## Processing Qualitative Features

For a continuous target variable, following processing is applied:
* Pre-processing of ordinals:
    - ordering modalities according to user-provided values
    - grouping modalities with less than `min_freq=3%` frequency into their closest modality (previous or next modality) according to target mean (train sample)
* Pre-processing of categoricals:
    - grouping modalities with less than `min_freq=3%` frequency into a dedicated one (train sample)
    - ordering modalities according to target mean (train sample)
* All combinations of up to `max_n_mod=5` modalities are sorted by Kruskall-'s T with the target variable (train sample)
* Robustness of each combination is put to test (dev sample) 
    - representativness of modalities (more than `min_freq/3=1.5%` frequency)
    - distinct target mean per consecutive modalities
    - no inversion of target means between train and dev modalities

For multiclass target variables, the binary-oriented processing steps are applied to each class of the target variable with a One vs Rest approach (except for one of the classes)

In [ ]:
from AutoCarver import ContinuousCarver, Features

# defining the features to carve
features = Features(categoricals=categorical_columns, ordinals=ordinal_columns)

# defining the carver
carver = ContinuousCarver(
    features=features,
    min_freq=0.03,
    max_n_mod=5,
    dropna=False,
    copy=False,
    verbose=False,
)

# carving train data and testing robustness on dev data
x_train = carver.fit_transform(x_train, y_train, X_dev=x_dev, y_dev=y_dev)

In [ ]:
# version = "018"
# carver.save(f"model/amount/{version}_carver.json", light_mode=True)

In [8]:
from AutoCarver import ContinuousCarver

version = "018"
carver = ContinuousCarver.load(f"model/amount/{version}_carver.json")
carver.summary

content  \
feature                          label                                                      
Categorical('ACTIVIT2')          0                                                   ACT2   
                                 1      [ACT5, ACT4, ACT6, ACT7, ACT9, ACT3, ACT8, __O...   
                                 2                                                   ACT1   
Categorical('VOCATION')          0              [VOC7, VOC5, VOC3, VOC2, __OTHER__, VOC8]   
                                 1                                                   VOC1   
...                                                                                   ...   
Ordinal('total_surface_5y')      0                                     [<10ha, Aucun feu]   
                                 1                                     [20-50ha, 10-20ha]   
                                 2                          [100-200ha, >200ha, 50-100ha]   
Ordinal('fire_extinction_rates') 0                      [<50%, 50-70%, 70-85%, Aucun feu]   
                                 1                                                   >85%   

                                        target_mean  frequency  
feature                          label                          
Categorical('ACTIVIT2')          0        69.750884   0.037225  
                                 1       157.138481   0.189379  
                                 2       189.147584   0.773396  
Categorical('VOCATION')          0        41.628900   0.285286  
                                 1        69.750884   0.037225  
...                                             ...        ...  
Ordinal('total_surface_5y')      0       167.873441   0.400648  
                                 1       191.128641   0.268860  
                                 2       181.535885   0.330492  
Ordinal('fire_extinction_rates') 0       156.601832   0.743216  
                                 1       242.430061   0.256784  

[742 rows x 3 columns]

## TODO: remove this

In [ ]:
# from AutoCarver import Features

# features = Features(quantitatives=numerical_columns)

In [ ]:
# from AutoCarver import ContinuousCarver

# carver_numericals = ContinuousCarver(
#     features=features,
#     min_freq=0.10,
#     dropna=False,
#     copy=False,
#     verbose=False,
# )
# x_train = carver_numericals.fit_transform(x_train, y_train, X_dev=x_dev, y_dev=y_dev)

In [ ]:
# version = "018"
# carver_numericals.save(f"model/amount/{version}_carver_numericals.json", light_mode=True)

In [7]:
from AutoCarver import ContinuousCarver

version = "018"
carver_numericals = ContinuousCarver.load(
    f"model/amount/{version}_carver_numericals.json"
)
carver_numericals.summary

content  target_mean  \
feature                     label                                          
Quantitative('ANCIENNETE')  0                 x <= 3.00e+00   178.012677   
                            1      3.00e+00 < x <= 5.00e+00   144.529584   
                            2      5.00e+00 < x <= 1.00e+01   172.646089   
                            3                  1.00e+01 < x   222.968206   
Quantitative('TYPBAT2')     0                 x <= 0.00e+00   113.304806   
...                                                     ...          ...   
Quantitative('KAPITAL_MAX') 0                 x <= 1.50e+04    41.017533   
                            1      1.50e+04 < x <= 6.25e+04    50.506749   
                            2      6.25e+04 < x <= 1.25e+05   134.933830   
                            3      1.25e+05 < x <= 2.25e+05   162.023674   
                            4                  2.25e+05 < x   500.578287   

                                   frequency  
feature                     label             
Quantitative('ANCIENNETE')  0       0.408615  
                            1       0.150625  
                            2       0.281044  
                            3       0.159716  
Quantitative('TYPBAT2')     0       0.316353  
...                                      ...  
Quantitative('KAPITAL_MAX') 0       0.271653  
                            1       0.145007  
                            2       0.112337  
                            3       0.268072  
                            4       0.202931  

[356 rows x 3 columns]

## Applying processing to samples

In [9]:
x_train = carver.transform(x_train)
x_dev = carver.transform(x_dev)
# x_train = carver_numericals.transform(x_train)
# x_dev = carver_numericals.transform(x_dev)

In [ ]:
# renamed_numerical_columns = [c + "_raw" for c in numerical_columns]
# x_train[renamed_numerical_columns] = x_train_copy[numerical_columns]
# x_dev[renamed_numerical_columns] = x_dev_copy[numerical_columns]
# numerical_columns = renamed_numerical_columns[:]

<tmp>/ipykernel_23052\1709348516.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  x_train[renamed_numerical_columns] = x_train_copy[numerical_columns]
<tmp>/ipykernel_23052\1709348516.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  x_train[renamed_numerical_columns] = x_train_copy[numerical_columns]
<tmp>/ipykernel_23052\1709348516.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once usi

Sanity check on X_TEST

In [11]:
import pandas as pd

data_path = "../data/"
oos = pd.read_csv(data_path + "test_input_5qJzHrr.csv")
oos = proc.transform(oos)
oos = carver.transform(oos)

<tmp>/ipykernel_20056\1035544443.py:4: DtypeWarning: Columns (16,17,29,30,31,126,128,129,132,133,135,138,371) have mixed types. Specify dtype option on import or set low_memory=False.
  oos = pd.read_csv(data_path+'test_input_5qJzHrr.csv')


In [12]:
oos.shape, x_train.shape, x_dev.shape, data.shape

((95852, 560), (306888, 562), (76722, 562), (383610, 562))

## Adding Feature Processing & Predictions from Frequency Model

In [13]:
# loading processed data
version = "012"
merged = pd.read_csv(data_path + f"frequency_{version}.csv")
merged.set_index("ID", inplace=True)

# merging with train and dev data
x_train = x_train.join(merged[[c for c in merged.columns if c not in x_train.columns]])
x_dev = x_dev.join(merged[[c for c in merged.columns if c not in x_dev.columns]])

<tmp>/ipykernel_20056\3424482877.py:3: DtypeWarning: Columns (16,30,31,33) have mixed types. Specify dtype option on import or set low_memory=False.
  merged = pd.read_csv(data_path+f'frequency_{version}.csv')


In [14]:
# loading processed OOS sample
version = "012"
oos.set_index("ID", inplace=True)
oos_freq = pd.read_csv(data_path + f"oos_frequency_{version}.csv")
oos_freq.set_index("ID", inplace=True)

# merging with OOS sample
oos = oos.join(oos_freq[[c for c in oos_freq.columns if c not in oos.columns]])

<tmp>/ipykernel_20056\2381959545.py:4: DtypeWarning: Columns (17,18,30,31,32,34,86,88,127,129,130,133,134,136,139,372) have mixed types. Specify dtype option on import or set low_memory=False.
  oos_freq = pd.read_csv(data_path+f"oos_frequency_{version}.csv")


In [15]:
x_train.shape, x_dev.shape, oos.shape

((306888, 1402), (76722, 1402), (95852, 1393))

## Weighting

Here, weighting is computed as the absoule error of prediction from the Claim Frequency model. This will help the Claim Amount model focus on the frequency's model errors and, fix some of it shortcomings

In [22]:
w_train = (x_train["pred_sum"] - x_train["FREQ"]).abs()
w_dev = (x_dev["pred_sum"] - x_dev["FREQ"]).abs()

# Feature Selection

## Selecting Quantitative Features

First:
* Association with the target variable:
    - association is measured as Distance Correlation between features and target
    - features with associations lower than the `threshold=0.002` of Distance Correlation are removed
* Features are sorted according to their association with the target variable
* Association between features:
    - association is measured using Spearman's rho between quantitative features
    - features with associations greater than the `threshold=0.9` of Spearman's rho (in absolute value), with a feature more associated to the target variable, are removed
* The `n_best_per_type=100` best quantitative features are kept

Second:
* Association with the target variable:
    - association is measured as Spearman's rho between features and target
    - features with associations lower than the `threshold=0.005` of Spearman's rho are removed
* Features are sorted according to their association with the target variable
* Association between features:
    - association is measured using Spearman's rho between quantitative features
    - features with associations greater than the `threshold=0.9` of Spearman's rho (in absolute value), with a feature more associated to the target variable, are removed
* The `n_best_per_type=100` best quantitative features are kept

Repeating this process ensures the best features are kept, and lets the model decide which feature to use.

In [18]:
from AutoCarver import Features
from AutoCarver.selectors import (
    RegressionSelector,
    SpearmanFilter,
    SpearmanMeasure,
    DistanceMeasure,
)

# defining the features to select
quantitative_features = Features(quantitatives=numerical_columns)
print("number of quantitative features:", len(quantitative_features))

# defining the target association measures and threshold used to select features
measures = [DistanceMeasure(threshold=0.002), SpearmanMeasure(threshold=0.005)]

# defining the inter-feature association measures and threshold used to filter out features
filters = [SpearmanFilter(threshold=0.9)]

# initiating the selector
selector = RegressionSelector(
    features=quantitative_features,
    measures=measures,
    filters=filters,
    n_best_per_type=100,
    max_num_features_per_chunk=1000,
    verbose=True,
)

# feature selection on train data
best_quantitative_features = selector.select(x_train, y_train)
print("number of selected quantitative features:", len(best_quantitative_features))

number of quantitative features: 219
 [RegressionSelector] Selected Quantitative Features 


,feature,Nan,Mode,DistanceMeasure,SpearmanMeasure,DistanceRank,SpearmanRank,SpearmanFilter,SpearmanWith
59,Quantitative('SURFACE11'),0.2289,0.0661,-0.0326,0.0470,0.0000,nan,0.9309,SURFACE1
46,Quantitative('KAPITAL32'),0.0000,0.3483,-0.0265,0.0556,1.0000,0.0000,0.0000,itself
65,Quantitative('SURFACE17'),0.0000,0.8319,-0.0252,0.0321,2.0000,11.0000,0.4784,SURFACE10
218,Quantitative('KAPITAL_MAX'),0.0000,0.2486,-0.0237,0.0521,3.0000,nan,0.9254,KAPITAL_SUM
58,Quantitative('SURFACE10'),0.0195,0.5234,-0.0208,0.0502,4.0000,2.0000,0.6123,KAPITAL32
70,Quantitative('NBBAT1'),0.0000,0.1215,-0.0199,0.0424,5.0000,nan,0.9998,NBBAT4
35,Quantitative('KAPITAL21'),0.0142,0.5734,-0.0189,0.0460,6.0000,6.0000,0.7055,KAPITAL32
24,Quantitative('KAPITAL10'),0.0000,0.5995,-0.0179,0.0253,7.0000,17.0000,0.7041,KAPITAL_SUM
106,Quantitative('CA3_CA_MEAN'),0.9603,0.0360,-0.0153,0.0144,8.0000,24.0000,-0.5501,CA3_CA_TOT
96,Quantitative('EQUIPEMENT6'),0.0000,0.1003,-0.0151,0.0351,9.0000,10.0000,0.3946,KAPITAL21


number of selected quantitative features: 59


## Selecting Qualitative Features

* Association with the target variable:
    - association is measured as Kruskal-Wallis' test statistic between the distributions of target for distinct feature classes
    - features with associations lower than the `threshold=0.005` of Kruskal-Wallis' test statistic are removed
* Features are sorted according to their association with the target variable
* Association between features:
    - association is measured using Cramér's V between qualitative features
    - features with associations greater than the `threshold=0.9` of Cramér's V, with a feature more associated to the target variable, are removed
* The `n_best_per_type=100` best quantitative features are kept

First, features processed to maximize association with Claim Amount are selected.
Then, features processed to maximize association with Claim Frequency are selected.
This helps avoid only one processing per feature being selected, but keeps several versions and lets the model decided which one to use 

In [19]:
from AutoCarver import Features
from AutoCarver.selectors import KruskalMeasure, CramervFilter


# defining the features to select
qualitative_features = Features(
    carver.features.versions
    + [
        "DEROG13_formatted",
        "DEROG8_formatted",
        "DEROG3_formatted",
        "DEROG16_formatted",
        "DEROG14_formatted",
    ]
)
print("number of qualitative features:", len(qualitative_features))

# defining the target association measures and threshold used to select features
measures = [KruskalMeasure(threshold=0.005)]

# defining the inter-feature association measures and threshold used to filter out features
filters = [CramervFilter(threshold=0.9)]

# initiating the selector
selector = RegressionSelector(
    features=qualitative_features,
    measures=measures,
    filters=filters,
    n_best_per_type=100,
    max_num_features_per_chunk=10000,
    verbose=True,
)

# feature selection on train data
best_qualitative_features = selector.select(x_train, y_train)
print("number of selected qualitative features:", len(best_qualitative_features))

number of qualitative features: 279


<home>/AppData\Local\pypoetry\Cache\virtualenvs\caa-challenge-frequency-BKzrcC_y-py3.11\Lib\site-packages\AutoCarver\selectors\measures\quantitative_measures.py:57: SmallSampleWarning: One or more sample arguments is too small; all returned values will be NaN. See documentation for sample size requirements.
  kw = kruskal(*tuple(x[(~nans) & (y == y_value)] for y_value in y_values))


 [RegressionSelector] Selected Qualitative Features 


,feature,Nan,Mode,KruskalMeasure,KruskalRank,CramervFilter,CramervWith
269,Categorical('SURFACE4'),0.0000,0.4509,911.1613,0.0000,0.0000,itself
266,Categorical('TAILLE1'),0.0000,0.3302,907.8587,1.0000,0.6365,SURFACE4
267,Categorical('TAILLE2'),0.0000,0.3991,749.9631,2.0000,0.6545,TAILLE1
12,Categorical('KAPITAL40'),0.0000,0.5733,560.8906,3.0000,0.5327,TAILLE1
1,Categorical('VOCATION'),0.0000,0.6374,518.6191,4.0000,0.5270,KAPITAL40
80,Categorical('TYPERS'),0.0000,0.6654,453.8045,5.0000,0.5049,TAILLE1
13,Categorical('KAPITAL41'),0.0000,0.8321,389.4002,6.0000,0.3392,KAPITAL40
14,Categorical('KAPITAL42'),0.0000,0.9558,291.8427,7.0000,0.3524,KAPITAL41
20,Categorical('RISK11'),0.0000,0.5901,267.0361,8.0000,0.3972,TAILLE1
11,Categorical('KAPITAL37'),0.0000,0.9530,176.5654,9.0000,0.2613,TAILLE2


number of selected qualitative features: 29


In [20]:
from AutoCarver import Features
from AutoCarver.selectors import KruskalMeasure, CramervFilter


# defining the features to select
qualitative_features2 = Features([c for c in x_train.columns if "__y=" in c])
print("number of qualitative features:", len(qualitative_features2))

# defining the target association measures and threshold used to select features
measures = [KruskalMeasure(threshold=0.005)]

# defining the inter-feature association measures and threshold used to filter out features
filters = [CramervFilter(threshold=0.9)]

# initiating the selector
selector = RegressionSelector(
    features=qualitative_features2,
    measures=measures,
    filters=filters,
    n_best_per_type=100,
    max_num_features_per_chunk=10000,
    verbose=True,
)

# feature selection on train data
best_qualitative_features2 = selector.select(x_train, y_train)
print("number of selected qualitative features:", len(best_qualitative_features2))

number of qualitative features: 833


<home>/AppData\Local\pypoetry\Cache\virtualenvs\caa-challenge-frequency-BKzrcC_y-py3.11\Lib\site-packages\AutoCarver\selectors\measures\quantitative_measures.py:57: SmallSampleWarning: One or more sample arguments is too small; all returned values will be NaN. See documentation for sample size requirements.
  kw = kruskal(*tuple(x[(~nans) & (y == y_value)] for y_value in y_values))


 [RegressionSelector] Selected Qualitative Features 


,feature,Nan,Mode,KruskalMeasure,KruskalRank,CramervFilter,CramervWith
12,Categorical('KAPITAL32__y=1'),0.0000,0.5732,759.3459,0.0000,0.0000,itself
130,Categorical('KAPITAL12__y=2'),0.0000,0.4928,691.8030,1.0000,0.6274,KAPITAL32__y=1
14,Categorical('SURFACE2__y=1'),0.0000,0.7116,681.8810,2.0000,0.5444,KAPITAL32__y=1
824,Categorical('SURFACE4__y=2'),0.0000,0.8255,678.8892,3.0000,0.7222,SURFACE2__y=1
136,Categorical('KAPITAL32__y=2'),0.0000,0.7944,671.8154,4.0000,0.5896,KAPITAL32__y=1
6,Categorical('KAPITAL12__y=1'),0.0000,0.5470,640.3088,5.0000,0.8325,KAPITAL12__y=2
23,Categorical('NBBAT4__y=1'),0.0000,0.7511,562.7446,6.0000,0.6058,SURFACE2__y=1
341,Categorical('KAPITAL40__y=2'),0.0000,0.5733,560.8906,7.0000,0.6405,KAPITAL32__y=1
821,Categorical('TAILLE1__y=2'),0.0000,0.8703,560.8321,8.0000,0.7231,SURFACE4__y=2
147,Categorical('NBBAT4__y=2'),0.0000,0.8657,489.3570,9.0000,0.6841,NBBAT4__y=1


number of selected qualitative features: 56


In [ ]:
# import json

# version = "018"

# # listing selected features
# best_features = best_quantitative_features.versions + best_qualitative_features.versions + best_qualitative_features2.versions

# # saving best features
# with open(f"model/amount/{version}_best_features.json", "w", encoding="utf-8") as json_file:
#     json.dump(best_features, json_file)

In [25]:
import json

version = "018"

# laoding best features
with open(
    f"model/amount/{version}_best_features.json", "r", encoding="utf-8"
) as json_file:
    best_features = json.load(json_file)

# XGBoost Modeling

## Hyperparameter Fine-Tuning with Bayesian Optimization

### Default XGBoost hyperparameters

In the following code, the XGBoost model's hyperparameters are fine-tuned using Optuna's implementation of Bayesian Optimization. This tuning process is designed to minimize the log loss on a development set in a multi-class classification context.

* A `xboost.XGBRegressor` is used with a `reg:tweedie` objective function (for the target to be correctly defined, it's floored to 0)
* The weighted train sample is used to fit the model with the suggested hyperparameters
* The weighted dev sample is used to evaluate model performance
* The metric used for optimization is the multiclass logarithmic loss, computed with ``sklearn.metrics.root_mean_squared_error``
* The default parameter ranges for optimization are defined as:

````python
XGB_PARAMS_RANGES = {
    "n_estimators": (100, 600),
    "learning_rate": (1e-4, 1),
    "max_depth": (1, 10),
    "min_child_weight": (1, 20),
    "subsample": (0.2, 1),
    "colsample_bytree": (0.2, 1),
    "colsample_bylevel": (0.2, 1),
    "gamma": (0, 10),
    "alpha": (0, 10),
    "lambda": (0, 10),
}
````

Key takeaways:
 * The Bayesian model is trained to predict output performance on dev sample from input hyperparameters
 * This implementation of the objective function ensures robustness of performances for selected hyperparameters

In [ ]:
import optuna
from utils.objectives import get_regression_objective

# number of trials
N_TRIALS = 400

# filtering out negative valuues (necessary for tweedie)
y_transform = lambda u: u.where(u >= 0, 0)

# defining the objective function of bayesian model
objective = get_regression_objective(
    x_train[best_features],
    y_transform(y_train),
    x_dev[best_features],
    y_transform(y_dev),
    objective="reg:tweedie",
    w_train=w_train,
    w_dev=w_dev,
)  # , sorted_features=sorted_features)

# creating optuna study
study = optuna.create_study(direction="minimize")

# optimizing the objective function
study.optimize(objective, n_trials=N_TRIALS)

[I 2025-04-28 18:01:03,117] A new study created in memory with name: no-name-dc1b67ad-541c-4bbd-9d61-d50e366ab63f
[I 2025-04-28 18:01:10,896] Trial 0 finished with value: 14692.835942750668 and parameters: {'objective': 'reg:tweedie', 'n_estimators': 153, 'learning_rate': 0.8133374157899137, 'max_depth': 7, 'min_child_weight': 5, 'subsample': 0.8939382847031314, 'colsample_bytree': 0.6669998962050558, 'colsample_bylevel': 0.3054581845042118, 'gamma': 3.244926970774479, 'alpha': 1.2115020146217037, 'lambda': 7.31049564957734, 'tweedie_variance_power': 1.2306402475429774}. Best is trial 0 with value: 14692.835942750668.
[I 2025-04-28 18:01:22,948] Trial 1 finished with value: 183110.0314924324 and parameters: {'objective': 'reg:tweedie', 'n_estimators': 455, 'learning_rate': 0.7596173439117877, 'max_depth': 4, 'min_child_weight': 3, 'subsample': 0.5338943623129025, 'colsample_bytree': 0.2835620564009269, 'colsample_bylevel': 0.9479100718199192, 'gamma': 5.208323152291175, 'alpha': 8.6699

In [49]:
study.optimize(objective, n_trials=N_TRIALS)

[I 2025-04-28 16:08:48,600] Trial 209 finished with value: 6612.282248840778 and parameters: {'objective': 'reg:tweedie', 'n_estimators': 163, 'learning_rate': 0.1078364467048612, 'max_depth': 1, 'min_child_weight': 15, 'subsample': 0.3416023825783894, 'colsample_bytree': 0.821569192981221, 'colsample_bylevel': 0.525784164249208, 'gamma': 2.0065827120687354, 'alpha': 3.097507747299344, 'lambda': 6.640999238173501, 'tweedie_variance_power': 1.7435529123199167}. Best is trial 177 with value: 6611.503285439611.
[I 2025-04-28 16:08:54,177] Trial 210 finished with value: 6612.65806226705 and parameters: {'objective': 'reg:tweedie', 'n_estimators': 163, 'learning_rate': 0.11038213409652715, 'max_depth': 1, 'min_child_weight': 16, 'subsample': 0.35004932311972575, 'colsample_bytree': 0.8746546212506489, 'colsample_bylevel': 0.5519930304754719, 'gamma': 1.9975135828171309, 'alpha': 3.1162303105781706, 'lambda': 7.479055161304981, 'tweedie_variance_power': 1.739433398598517}. Best is trial 177 

In [50]:
study.optimize(objective, n_trials=N_TRIALS)

[I 2025-04-28 16:30:17,671] Trial 409 finished with value: 6611.785222486308 and parameters: {'objective': 'reg:tweedie', 'n_estimators': 209, 'learning_rate': 0.13170480133020018, 'max_depth': 1, 'min_child_weight': 2, 'subsample': 0.2555378761363402, 'colsample_bytree': 0.8680155773454611, 'colsample_bylevel': 0.6402298774248129, 'gamma': 3.543561185302407, 'alpha': 2.530250356505782, 'lambda': 6.663561121765694, 'tweedie_variance_power': 1.7005623737255102}. Best is trial 382 with value: 6610.474845793164.
[I 2025-04-28 16:30:24,517] Trial 410 finished with value: 7140.669330601935 and parameters: {'objective': 'reg:tweedie', 'n_estimators': 210, 'learning_rate': 0.5542974363881137, 'max_depth': 1, 'min_child_weight': 2, 'subsample': 0.22817421264898807, 'colsample_bytree': 0.8766138526810967, 'colsample_bylevel': 0.6365772713835918, 'gamma': 3.6607768043411255, 'alpha': 2.5576203440667946, 'lambda': 7.048168269688783, 'tweedie_variance_power': 1.699720814678943}. Best is trial 382 

In [89]:
from utils.objectives import get_best_regression_model

selected_features, xgb = get_best_regression_model(
    study.best_params,
    x_train[best_features],
    y_transform(y_train),
    x_dev[best_features],
    y_dev,
    w_train=w_train,
    w_dev=w_dev,
    train_on_full=False,
)

used features: ['SURFACE11', 'KAPITAL32', 'SURFACE17', 'KAPITAL_MAX', 'SURFACE10', 'NBBAT1', 'KAPITAL21', 'KAPITAL10', 'CA3_CA_MEAN', 'EQUIPEMENT6', 'SURFACE7', 'KAPITAL24', 'NBSINSTRT', 'CA3', 'KAPITAL27', 'SURFACE8', 'KAPITAL23', 'KAPITAL3', 'KAPITAL14', 'NBBAT8', 'DEROG11', 'RISK5', 'RISK4', 'RISK1', 'RISK3', 'ALTITUDE_5_ALT_TOT', 'NBSINCONJ', 'ALTITUDE_2_ALT_TOT', 'NBBAT6', 'CARACT5', 'KAPITAL26', 'DEROG10', 'NBBAT9', 'KAPITAL28', 'CARACT2', 'KAPITAL8', 'EQUIPEMENT1', 'KAPITAL18', 'TAILLE3', 'ALTITUDE_4_num', 'IND_Y5_Y6_num', 'KAPITAL25', 'ALTITUDE_3_num', 'IND_Y5_Y6_IND_SNV', 'ANCIENNETE', 'DEROG1', 'ALTITUDE_REGION', 'CA3_CA_TOT', 'KAPITAL17', 'DEROG6', 'KAPITAL16', 'KAPITAL_SUM', 'SURFACE1', 'KAPITAL12', 'NBBAT4', 'NBBAT10', 'RISK2', 'ALTITUDE_2_num', 'IND_Y5_Y6_IND', 'SURFACE4', 'TAILLE1', 'TAILLE2', 'KAPITAL40', 'VOCATION', 'TYPERS', 'KAPITAL41', 'KAPITAL42', 'RISK11', 'KAPITAL37', 'KAPITAL43', 'ACTIVIT2', 'DEROG4', 'DEROG5', 'CARACT4', 'AN_EXERC', 'COEFASS', 'DEROG2', 'ZONE_R

In [51]:
from utils.objectives import get_best_regression_model

selected_features, xgb = get_best_regression_model(
    study.best_params,
    x_train[best_features],
    y_transform(y_train),
    x_dev[best_features],
    y_dev,
    train_on_full=False,
)

used features: ['SURFACE11', 'KAPITAL32', 'SURFACE17', 'KAPITAL_MAX', 'SURFACE10', 'NBBAT1', 'KAPITAL21', 'KAPITAL10', 'CA3_CA_MEAN', 'EQUIPEMENT6', 'SURFACE7', 'KAPITAL24', 'NBSINSTRT', 'CA3', 'KAPITAL27', 'SURFACE8', 'KAPITAL23', 'KAPITAL3', 'KAPITAL14', 'NBBAT8', 'DEROG11', 'RISK5', 'RISK4', 'RISK1', 'RISK3', 'ALTITUDE_5_ALT_TOT', 'NBSINCONJ', 'ALTITUDE_2_ALT_TOT', 'NBBAT6', 'CARACT5', 'KAPITAL26', 'DEROG10', 'NBBAT9', 'KAPITAL28', 'CARACT2', 'KAPITAL8', 'EQUIPEMENT1', 'KAPITAL18', 'TAILLE3', 'ALTITUDE_4_num', 'IND_Y5_Y6_num', 'KAPITAL25', 'ALTITUDE_3_num', 'IND_Y5_Y6_IND_SNV', 'ANCIENNETE', 'DEROG1', 'ALTITUDE_REGION', 'CA3_CA_TOT', 'KAPITAL17', 'DEROG6', 'KAPITAL16', 'KAPITAL_SUM', 'SURFACE1', 'KAPITAL12', 'NBBAT4', 'NBBAT10', 'RISK2', 'ALTITUDE_2_num', 'IND_Y5_Y6_IND', 'SURFACE4', 'TAILLE1', 'TAILLE2', 'KAPITAL40', 'VOCATION', 'TYPERS', 'KAPITAL41', 'KAPITAL42', 'RISK11', 'KAPITAL37', 'KAPITAL43', 'ACTIVIT2', 'DEROG4', 'DEROG5', 'CARACT4', 'AN_EXERC', 'COEFASS', 'DEROG2', 'ZONE_R

### LEGACY: OLDER MODELS

In [38]:
from utils.objectives import get_best_regression_model

selected_features, xgb = get_best_regression_model(
    study.best_params,
    x_train[best_features],
    y_transform(y_train),
    x_dev[best_features],
    y_dev,
    train_on_full=False,
)

used features: ['SURFACE11_raw', 'KAPITAL32_raw', 'SURFACE17_raw', 'KAPITAL_MAX_raw', 'SURFACE10_raw', 'NBBAT1_raw', 'KAPITAL21_raw', 'KAPITAL10_raw', 'KAPITAL24_raw', 'SURFACE7_raw', 'EQUIPEMENT6_raw', 'KAPITAL27_raw', 'CA1_CA_TOT_raw', 'NBSINSTRT_raw', 'NBBAT2_raw', 'KAPITAL23_raw', 'KAPITAL14_raw', 'NBBAT8_raw', 'RISK5_raw', 'RISK4_raw', 'RISK1_raw', 'RISK3_raw', 'ALTITUDE_3_ALT_TOT_raw', 'NBSINCONJ_raw', 'CARACT5_raw', 'KAPITAL3_raw', 'KAPITAL25_raw', 'NBBAT6_raw', 'ALTITUDE_2_ALT_TOT_raw', 'KAPITAL17_raw', 'DEROG11_raw', 'KAPITAL28_raw', 'EQUIPEMENT1_raw', 'ALTITUDE_5_ALT_TOT_raw', 'CARACT2_raw', 'KAPITAL8_raw', 'ALTITUDE_3_num_raw', 'KAPITAL20_raw', 'ALTITUDE_REGION_raw', 'ALTITUDE_4_num_raw', 'TAILLE3_raw', 'KAPITAL16_raw', 'NBBAT9_raw', 'DEROG10_raw', 'LOG_APA3_num_raw', 'IND_Y5_Y6_num_raw', 'DEROG1_raw', 'DEROG7_raw', 'SURFACE13_raw', 'EQUIPEMENT4_raw', 'KAPITAL_SUM_raw', 'KAPITAL12_raw', 'SURFACE1_raw', 'NBBAT4_raw', 'CA3_CA_MEAN_raw', 'RISK2_raw', 'ALTITUDE_TOT_raw', 'KAPITA

In [34]:
from utils.objectives import get_best_regression_model

selected_features, xgb = get_best_regression_model(
    study.best_params,
    x_train[best_features],
    y_transform(y_train),
    x_dev[best_features],
    y_dev,
    train_on_full=False,
)

used features: ['SURFACE11_raw', 'KAPITAL32_raw', 'SURFACE17_raw', 'KAPITAL_MAX_raw', 'SURFACE10_raw', 'NBBAT1_raw', 'KAPITAL21_raw', 'KAPITAL10_raw', 'KAPITAL24_raw', 'SURFACE7_raw', 'EQUIPEMENT6_raw', 'KAPITAL27_raw', 'CA1_CA_TOT_raw', 'NBSINSTRT_raw', 'NBBAT2_raw', 'KAPITAL23_raw', 'KAPITAL14_raw', 'NBBAT8_raw', 'RISK5_raw', 'RISK4_raw', 'RISK1_raw', 'RISK3_raw', 'ALTITUDE_3_ALT_TOT_raw', 'NBSINCONJ_raw', 'CARACT5_raw', 'KAPITAL3_raw', 'KAPITAL25_raw', 'NBBAT6_raw', 'ALTITUDE_2_ALT_TOT_raw', 'KAPITAL17_raw', 'DEROG11_raw', 'KAPITAL28_raw', 'EQUIPEMENT1_raw', 'ALTITUDE_5_ALT_TOT_raw', 'CARACT2_raw', 'KAPITAL8_raw', 'ALTITUDE_3_num_raw', 'KAPITAL20_raw', 'ALTITUDE_REGION_raw', 'ALTITUDE_4_num_raw', 'TAILLE3_raw', 'KAPITAL16_raw', 'NBBAT9_raw', 'DEROG10_raw', 'LOG_APA3_num_raw', 'IND_Y5_Y6_num_raw', 'DEROG1_raw', 'DEROG7_raw', 'SURFACE13_raw', 'EQUIPEMENT4_raw', 'KAPITAL_SUM_raw', 'KAPITAL12_raw', 'SURFACE1_raw', 'NBBAT4_raw', 'CA3_CA_MEAN_raw', 'RISK2_raw', 'ALTITUDE_TOT_raw', 'KAPITA

In [138]:
from utils.objectives import get_best_regression_model

filtered_features = [
    f
    for f in best_features
    if f in sorted_features[: study.best_params["n_features_removed"]]
]

selected_features, xgb = get_best_regression_model(
    study.best_params,
    x_train[best_features],
    y_transform(y_train),
    x_dev[best_features],
    y_dev,
    train_on_full=False,
)

used features: ['SURFACE11', 'KAPITAL32', 'SURFACE17', 'SURFACE10', 'NBBAT1', 'KAPITAL21', 'KAPITAL12', 'KAPITAL10', 'KAPITAL24', 'SURFACE7', 'EQUIPEMENT6', 'KAPITAL27', 'CA1_CA_TOT', 'NBSINSTRT', 'NBBAT2', 'KAPITAL23', 'KAPITAL14', 'NBBAT8', 'RISK5', 'RISK4', 'RISK1', 'RISK3', 'ALTITUDE_3_ALT_TOT', 'NBSINCONJ', 'CARACT5', 'KAPITAL3', 'KAPITAL25', 'NBBAT6', 'ALTITUDE_2_ALT_TOT', 'KAPITAL17', 'DEROG11', 'KAPITAL28', 'EQUIPEMENT1', 'ALTITUDE_5_ALT_TOT', 'CARACT2', 'KAPITAL8', 'ALTITUDE_3_num', 'ALTITUDE_REGION', 'SURFACE1', 'NBBAT4', 'CA3_CA_MEAN', 'RISK2', 'ALTITUDE_TOT', 'TAILLE1', 'SURFACE4', 'TAILLE2', 'VOCATION', 'KAPITAL40', 'TYPERS', 'KAPITAL41', 'RISK11', 'KAPITAL42', 'KAPITAL37', 'KAPITAL43', 'DEROG4', 'ACTIVIT2', 'DEROG5', 'CARACT4', 'COEFASS', 'AN_EXERC', 'zone_vent_extinction_rate', 'DEROG2', 'DEROG3_formatted', 'total_surface_5y', 'ZONE', 'total_surface_2023', 'ZONE_REGION', 'total_surface_crossed', 'FRCH1', 'fire_extinction_rates', 'DEROG16_formatted', 'DEROG13_formatted', 

<home>/AppData\Local\pypoetry\Cache\virtualenvs\caa-challenge-frequency-BKzrcC_y-py3.11\Lib\site-packages\xgboost\core.py:158: UserWarning: [14:57:45] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-08cbc0333d8d4aae1-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "n_features_removed" } are not used.

  warnings.warn(smsg, UserWarning)


RMSE Train: 3740.239501953125
RMSE Dev:   7183.995911649211


In [118]:
from utils.objectives import get_best_regression_model

selected_features, xgb = get_best_regression_model(
    study.best_params,
    x_train[best_features],
    y_transform(y_train),
    x_dev[best_features],
    y_dev,
    train_on_full=False,
)

used features: ['SURFACE11', 'KAPITAL32', 'SURFACE17', 'SURFACE10', 'NBBAT1', 'KAPITAL21', 'KAPITAL12', 'KAPITAL10', 'KAPITAL24', 'SURFACE7', 'EQUIPEMENT6', 'KAPITAL27', 'CA1_CA_TOT', 'NBSINSTRT', 'NBBAT2', 'KAPITAL23', 'KAPITAL14', 'NBBAT8', 'RISK5', 'RISK4', 'RISK1', 'RISK3', 'ALTITUDE_3_ALT_TOT', 'NBSINCONJ', 'CARACT5', 'KAPITAL3', 'KAPITAL25', 'NBBAT6', 'ALTITUDE_2_ALT_TOT', 'KAPITAL17', 'DEROG11', 'KAPITAL28', 'EQUIPEMENT1', 'ALTITUDE_5_ALT_TOT', 'CARACT2', 'KAPITAL8', 'ALTITUDE_3_num', 'ALTITUDE_REGION', 'SURFACE1', 'NBBAT4', 'CA3_CA_MEAN', 'RISK2', 'ALTITUDE_TOT', 'TAILLE1', 'SURFACE4', 'TAILLE2', 'VOCATION', 'KAPITAL40', 'TYPERS', 'KAPITAL41', 'RISK11', 'KAPITAL42', 'KAPITAL37', 'KAPITAL43', 'DEROG4', 'ACTIVIT2', 'DEROG5', 'CARACT4', 'COEFASS', 'AN_EXERC', 'zone_vent_extinction_rate', 'DEROG2', 'DEROG3_formatted', 'total_surface_5y', 'ZONE', 'total_surface_2023', 'ZONE_REGION', 'total_surface_crossed', 'FRCH1', 'fire_extinction_rates', 'DEROG16_formatted', 'DEROG13_formatted', 

In [77]:
from utils.objectives import get_best_regression_model

selected_features, xgb = get_best_regression_model(
    study.best_params,
    x_train[best_features],
    y_transform(y_train),
    x_dev[best_features],
    y_transform(y_dev),
    w_train=w_train,
    w_dev=w_dev,
    train_on_full=False,
)

used features: ['SURFACE11', 'KAPITAL32', 'SURFACE17', 'SURFACE10', 'NBBAT1', 'KAPITAL21', 'KAPITAL12', 'KAPITAL10', 'KAPITAL24', 'SURFACE7', 'EQUIPEMENT6', 'KAPITAL27', 'CA1_CA_TOT', 'NBSINSTRT', 'NBBAT2', 'KAPITAL23', 'KAPITAL14', 'NBBAT8', 'RISK5', 'RISK4', 'RISK1', 'RISK3', 'ALTITUDE_3_ALT_TOT', 'NBSINCONJ', 'CARACT5', 'KAPITAL3', 'KAPITAL25', 'NBBAT6', 'ALTITUDE_2_ALT_TOT', 'KAPITAL17', 'DEROG11', 'KAPITAL28', 'EQUIPEMENT1', 'ALTITUDE_5_ALT_TOT', 'CARACT2', 'KAPITAL8', 'ALTITUDE_3_num', 'ALTITUDE_REGION', 'SURFACE1', 'NBBAT4', 'CA3_CA_MEAN', 'RISK2', 'ALTITUDE_TOT', 'TAILLE1', 'SURFACE4', 'TAILLE2', 'VOCATION', 'KAPITAL40', 'TYPERS', 'KAPITAL41', 'RISK11', 'KAPITAL42', 'KAPITAL37', 'KAPITAL43', 'DEROG4', 'ACTIVIT2', 'DEROG5', 'CARACT4', 'COEFASS', 'AN_EXERC', 'zone_vent_extinction_rate', 'DEROG2', 'DEROG3_formatted', 'total_surface_5y', 'ZONE', 'total_surface_2023', 'ZONE_REGION', 'total_surface_crossed', 'FRCH1', 'fire_extinction_rates', 'DEROG16_formatted', 'DEROG13_formatted', 

In [59]:
from utils.objectives import get_best_regression_model

selected_features, xgb = get_best_regression_model(
    study.best_params,
    x_train[best_features],
    y_transform(y_train),
    x_dev[best_features],
    y_transform(y_dev),
    w_train=w_train,
    w_dev=w_dev,
    train_on_full=False,
)

used features: ['SURFACE11', 'KAPITAL32', 'SURFACE17', 'SURFACE10', 'NBBAT1', 'KAPITAL21', 'KAPITAL12', 'KAPITAL10', 'KAPITAL24', 'SURFACE7', 'EQUIPEMENT6', 'KAPITAL27', 'CA1_CA_TOT', 'NBSINSTRT', 'NBBAT2', 'KAPITAL23', 'KAPITAL14', 'NBBAT8', 'RISK5', 'RISK4', 'RISK1', 'RISK3', 'ALTITUDE_3_ALT_TOT', 'NBSINCONJ', 'CARACT5', 'KAPITAL3', 'KAPITAL25', 'NBBAT6', 'ALTITUDE_2_ALT_TOT', 'KAPITAL17', 'DEROG11', 'KAPITAL28', 'EQUIPEMENT1', 'ALTITUDE_5_ALT_TOT', 'CARACT2', 'KAPITAL8', 'ALTITUDE_3_num', 'ALTITUDE_REGION', 'SURFACE1', 'NBBAT4', 'CA3_CA_MEAN', 'RISK2', 'ALTITUDE_TOT', 'TAILLE1', 'SURFACE4', 'TAILLE2', 'VOCATION', 'KAPITAL40', 'TYPERS', 'KAPITAL41', 'RISK11', 'KAPITAL42', 'KAPITAL37', 'KAPITAL43', 'DEROG4', 'ACTIVIT2', 'DEROG5', 'CARACT4', 'COEFASS', 'AN_EXERC', 'zone_vent_extinction_rate', 'DEROG2', 'DEROG3_formatted', 'total_surface_5y', 'ZONE', 'total_surface_2023', 'ZONE_REGION', 'total_surface_crossed', 'FRCH1', 'fire_extinction_rates', 'DEROG16_formatted', 'DEROG13_formatted', 

In [68]:
from utils.objectives import get_best_regression_model

selected_features, xgb = get_best_regression_model(
    study.best_params,
    x_train[best_features],
    y_transform(y_train),
    x_dev[best_features],
    y_transform(y_dev),
    w_train=w_train,
    w_dev=w_dev,
)

used features: ['SURFACE11', 'KAPITAL32', 'SURFACE10', 'NBBAT1', 'KAPITAL10', 'KAPITAL24', 'CA1_CA_TOT', 'KAPITAL23', 'NBSINSTRT', 'ALTITUDE_2_ALT_TOT', 'RISK5', 'RISK4', 'RISK1', 'RISK3', 'ALTITUDE_3_ALT_TOT', 'KAPITAL14', 'NBBAT8', 'NBBAT2', 'DEROG11', 'KAPITAL20', 'TAILLE1', 'TAILLE2', 'SURFACE6__y=1', 'VOCATION', 'TYPERS__y=1', 'RISK11', 'KAPITAL42', 'ACTIVIT2', 'DEROG5', 'total_surface_crossed__y=1', 'CARACT4']
RMSE Train: 6662.251663068865
RMSE Dev:   6833.291115014657
RMSE Overall:   6696.48587866153


In [42]:
from utils.objectives import get_best_regression_model

selected_features, xgb = get_best_regression_model(
    study.best_params,
    x_train[best_features],
    y_transform(y_train),
    x_dev[best_features],
    y_transform(y_dev),
)

used features: ['KAPITAL32', 'NBBAT1', 'KAPITAL24', 'NBSINSTRT', 'RISK4', 'RISK1', 'RISK3', 'KAPITAL14', 'SURFACE16', 'KAPITAL8', 'EQUIPEMENT1', 'ALTITUDE_3_num', 'SURFACE6', 'TAILLE2', 'SURFACE6__y=2', 'VOCATION', 'KAPITAL41__y=1', 'ACTIVIT2', 'KAPITAL37', 'DEROG5', 'zone_vent_extinction_rate__y=2']
RMSE Train: 6653.316098374929
RMSE Dev:   6849.223554526236
RMSE Overall:   6692.0234504143355


In [58]:
from utils.objectives import get_best_regression_model

xgb = get_best_regression_model(
    study.best_params,
    x_train[best_features],
    y_train.where(y_train >= 0, 0),
    x_dev[best_features],
    y_dev.where(y_dev >= 0, 0),
)

used features: ['SURFACE11', 'KAPITAL32', 'SURFACE17', 'SURFACE10', 'KAPITAL12', 'KAPITAL21', 'KAPITAL10', 'EQUIPEMENT6', 'KAPITAL24', 'SURFACE7', 'CA1_CA_TOT', 'KAPITAL27', 'KAPITAL23', 'NBSINSTRT', 'ALTITUDE_2_ALT_TOT', 'RISK5', 'NBSINCONJ', 'KAPITAL3', 'KAPITAL14', 'SURFACE16', 'NBBAT10', 'IND_Y5_Y6_IND', 'ALTITUDE_1_num', 'SURFACE1', 'NBBAT13', 'CA3_CA_MEAN', 'KAPITAL28', 'SURFACE6', 'TAILLE1', 'TAILLE2', 'KAPITAL40', 'VOCATION', 'TYPERS__y=1', 'KAPITAL41', 'RISK11', 'KAPITAL42__y=2', 'KAPITAL43', 'KAPITAL37__y=2', 'DEROG4', 'total_surface_crossed__y=1']
RMSE Train: 6656.197455117935
RMSE Dev:   6851.5142527383905
RMSE Overall:   6695.580525922295


In [31]:
from utils.objectives import get_best_regression_model

xgb = get_best_regression_model(
    study.best_params,
    x_train[best_features],
    y_train.where(y_train >= 0, 0),
    x_dev[best_features],
    y_dev.where(y_dev >= 0, 0),
)

used features: ['SURFACE11', 'KAPITAL32', 'SURFACE17', 'SURFACE10', 'KAPITAL12', 'KAPITAL21', 'NBBAT1', 'KAPITAL10', 'EQUIPEMENT6', 'KAPITAL24', 'SURFACE7', 'CA1_CA_MEAN', 'CA1_CA_TOT', 'KAPITAL27', 'KAPITAL23', 'NBSINSTRT', 'ALTITUDE_2_ALT_TOT', 'RISK5', 'RISK4', 'RISK1', 'RISK3', 'NBSINCONJ', 'ALTITUDE_3_ALT_TOT', 'KAPITAL3', 'KAPITAL14', 'SURFACE16', 'NBBAT8', 'NBBAT2', 'CA_TOT', 'DEROG11', 'KAPITAL8', 'KAPITAL28', 'EQUIPEMENT1', 'DEROG10', 'CARACT2', 'IND_Y5_Y6_num', 'KAPITAL20', 'ALTITUDE_3_num', 'SURFACE6', 'TAILLE1', 'TAILLE2', 'SURFACE6__y=1', 'TAILLE1__y=2', 'KAPITAL40', 'SURFACE6__y=2', 'VOCATION', 'TYPERS__y=1', 'KAPITAL41__y=1', 'RISK11', 'KAPITAL42', 'KAPITAL43__y=1', 'ACTIVIT2', 'KAPITAL37', 'DEROG4', 'DEROG5', 'total_surface_crossed__y=1', 'AN_EXERC', 'CARACT4', 'ZONE', 'total_surface_5y__y=1', 'zone_vent_extinction_rate__y=2', 'pred_1', 'pred_2', 'pred_sum']
RMSE Train: 6656.131020415359
RMSE Dev:   6851.58630315876
RMSE Overall:   6695.28954870395


In [84]:
from utils.objectives import get_best_regression_model

xgb = get_best_regression_model(
    study.best_params,
    x_train[best_features],
    y_train.where(y_train >= 0, 0),
    x_dev[best_features],
    y_dev.where(y_dev >= 0, 0),
)

used features: ['SURFACE11', 'KAPITAL32', 'SURFACE17', 'SURFACE10', 'KAPITAL12', 'KAPITAL21', 'NBBAT1', 'KAPITAL10', 'EQUIPEMENT6', 'KAPITAL24', 'SURFACE7', 'CA1_CA_MEAN', 'CA1_CA_TOT', 'KAPITAL27', 'KAPITAL23', 'NBSINSTRT', 'ALTITUDE_2_ALT_TOT', 'RISK5', 'RISK4', 'RISK1', 'RISK3', 'NBSINCONJ', 'ALTITUDE_3_ALT_TOT', 'KAPITAL3', 'KAPITAL14', 'SURFACE16', 'NBBAT8', 'NBBAT2', 'CA_TOT', 'DEROG11', 'KAPITAL8', 'KAPITAL28', 'EQUIPEMENT1', 'DEROG10', 'CARACT2', 'IND_Y5_Y6_num', 'KAPITAL20', 'ALTITUDE_3_num', 'SURFACE6', 'TAILLE1', 'TAILLE2', 'SURFACE6__y=1', 'TAILLE1__y=2', 'KAPITAL40', 'SURFACE6__y=2', 'VOCATION', 'TYPERS__y=1', 'KAPITAL41__y=1', 'RISK11', 'KAPITAL42', 'KAPITAL43__y=1', 'ACTIVIT2', 'KAPITAL37', 'DEROG4', 'DEROG5', 'total_surface_crossed__y=1', 'AN_EXERC', 'CARACT4', 'ZONE', 'total_surface_5y__y=1', 'zone_vent_extinction_rate__y=2', 'pred_1', 'pred_2', 'pred_sum']
RMSE Train: 6653.455026967703
RMSE Dev:   6850.938824438842
RMSE Overall:   6693.234782286674


In [69]:
from utils.objectives import get_best_regression_model

xgb = get_best_regression_model(
    study.best_params,
    x_train[best_features],
    y_train.where(y_train >= 0, 0),
    x_dev[best_features],
    y_dev.where(y_dev >= 0, 0),
)

used features: ['SURFACE11', 'KAPITAL32', 'SURFACE17', 'SURFACE10', 'KAPITAL12', 'KAPITAL21', 'NBBAT1', 'KAPITAL10', 'EQUIPEMENT6', 'KAPITAL24', 'SURFACE7', 'CA1_CA_MEAN', 'CA1_CA_TOT', 'KAPITAL27', 'KAPITAL23', 'NBSINSTRT', 'ALTITUDE_2_ALT_TOT', 'RISK5', 'RISK4', 'RISK1', 'RISK3', 'NBSINCONJ', 'ALTITUDE_3_ALT_TOT', 'KAPITAL3', 'KAPITAL14', 'SURFACE16', 'NBBAT8', 'NBBAT2', 'CA_TOT', 'DEROG11', 'KAPITAL8', 'KAPITAL28', 'EQUIPEMENT1', 'DEROG10', 'CARACT2', 'IND_Y5_Y6_num', 'KAPITAL20', 'ALTITUDE_3_num', 'SURFACE6', 'TAILLE1', 'TAILLE2', 'SURFACE6__y=1', 'TAILLE1__y=2', 'KAPITAL40', 'SURFACE6__y=2', 'VOCATION', 'TYPERS__y=1', 'KAPITAL41__y=1', 'RISK11', 'KAPITAL42', 'KAPITAL43__y=1', 'ACTIVIT2', 'KAPITAL37', 'DEROG4', 'DEROG5', 'total_surface_crossed__y=1', 'AN_EXERC', 'CARACT4', 'ZONE', 'total_surface_5y__y=1', 'zone_vent_extinction_rate__y=2']
RMSE Train: 6656.363154451487
RMSE Dev:   6851.387568699358
RMSE Overall:   6695.832167136867


In [45]:
from utils.objectives import get_best_regression_model

xgb = get_best_regression_model(
    study.best_params, x_train[best_features], y_train, x_dev[best_features], y_dev
)

used features: ['SURFACE11', 'KAPITAL32', 'SURFACE17', 'SURFACE10', 'KAPITAL12', 'KAPITAL21', 'NBBAT1', 'KAPITAL10', 'EQUIPEMENT6', 'KAPITAL24', 'SURFACE7', 'CA1_CA_MEAN', 'CA1_CA_TOT', 'KAPITAL27', 'KAPITAL23', 'NBSINSTRT', 'ALTITUDE_2_ALT_TOT', 'RISK5', 'RISK4', 'RISK1', 'RISK3', 'NBSINCONJ', 'ALTITUDE_3_ALT_TOT', 'KAPITAL3', 'KAPITAL14', 'SURFACE16', 'NBBAT8', 'NBBAT2', 'CA_TOT', 'DEROG11', 'KAPITAL8', 'KAPITAL28', 'EQUIPEMENT1', 'DEROG10', 'CARACT2', 'IND_Y5_Y6_num', 'KAPITAL20', 'ALTITUDE_3_num', 'SURFACE6', 'TAILLE1', 'TAILLE2', 'SURFACE6__y=1', 'TAILLE1__y=2', 'KAPITAL40', 'SURFACE6__y=2', 'VOCATION', 'TYPERS__y=1', 'KAPITAL41__y=1', 'RISK11', 'KAPITAL42', 'KAPITAL43__y=1', 'ACTIVIT2', 'KAPITAL37', 'DEROG4', 'DEROG5', 'total_surface_crossed__y=1', 'AN_EXERC', 'CARACT4', 'ZONE', 'total_surface_5y__y=1', 'zone_vent_extinction_rate__y=2']
RMSE Train: 6655.419030852636
RMSE Dev:   6851.624607320323
RMSE Overall:   6695.322609566194


In [22]:
from utils.objectives import get_best_regression_model

xgb = get_best_regression_model(
    study.best_params, x_train[best_features], y_train, x_dev[best_features], y_dev
)

used features: ['SURFACE11', 'KAPITAL32', 'SURFACE17', 'SURFACE10', 'KAPITAL12', 'KAPITAL21', 'NBBAT1', 'KAPITAL10', 'EQUIPEMENT6', 'KAPITAL24', 'SURFACE7', 'CA1_CA_MEAN', 'CA1_CA_TOT', 'KAPITAL27', 'KAPITAL23', 'NBSINSTRT', 'ALTITUDE_2_ALT_TOT', 'RISK5', 'RISK4', 'RISK1', 'RISK3', 'NBSINCONJ', 'ALTITUDE_3_ALT_TOT', 'KAPITAL3', 'KAPITAL14', 'SURFACE16', 'NBBAT8', 'NBBAT2', 'CA_TOT', 'DEROG11', 'KAPITAL8', 'KAPITAL28', 'EQUIPEMENT1', 'DEROG10', 'CARACT2', 'IND_Y5_Y6_num', 'KAPITAL20', 'ALTITUDE_3_num', 'KAPITAL40', 'VOCATION', 'TYPERS', 'KAPITAL41', 'RISK11', 'KAPITAL42', 'KAPITAL43', 'ACTIVIT2', 'KAPITAL37', 'DEROG4', 'DEROG5', 'AN_EXERC', 'ZONE', 'DEROG2', 'ZONE_REGION', 'total_surface_crossed', 'zone_vent_extinction_rate', 'FRCH1', 'KAPITAL34', 'SURFACE4', 'TAILLE1', 'TAILLE2', 'CARACT4', 'COEFASS', 'total_surface_2023', 'total_surface_5y', 'fire_extinction_rates']
RMSE Train: 6655.088282457237
RMSE Dev:   6851.8193075803165
RMSE Overall:   6695.147604802015


### Saving fine-tuned model

In [ ]:
# version = "019"

# xgb.save_model(f"model/amount/{version}_xgboost.json")

# with open(f"model/amount/{version}_selected_features.json", "w", encoding="utf-8") as json_file:
#     json.dump(selected_features, json_file)

# Prediction on OOS

In [29]:
from xgboost import XGBRegressor


# loading trained model
version = "019"
xgb = XGBRegressor()
xgb.load_model(f"model/amount/{version}_xgboost.json")

# prediction on OOS
oos_pred = xgb.predict(oos[best_features])
oos_pred

array([183.34576 ,  68.01676 ,  58.0457  , ..., 208.28638 , 102.018585,
       276.44427 ], dtype=float32)

## Filling Output for Challenge Submission

In [ ]:
import pandas as pd

# reading Claim Frequency predictions
version = "025"
pred = pd.read_csv(f"predictions/{version}_pred.csv")

# adding in Claim Amount predictions
pred["CM"] = oos_pred

# computing the final prediction
pred["CHARGE"] = pred["FREQ"] * pred["CM"]  # * pred["ANNEE_ASSURANCE"]
pred.head()

,ID,ANNEE_ASSURANCE,FREQ,CM,CHARGE
0,383611,0.813699,1.094768,183.345947,200.721293
1,383612,1.000000,0.813482,68.016739,55.330376
2,383613,0.586301,0.468850,58.045670,27.214702
3,383614,1.000000,0.400128,156.917374,62.787102
4,383615,0.753425,0.441974,196.950485,87.046916


In [ ]:
# saving submission
version = "028"
pred.to_csv(f"predictions/{version}_pred.csv", index=False)